# CineMatch - Hybrid Movie Recommendation System

Interactive Walkthrough

This notebook walks through the recommendation engine step by step. It is built on the SVD + Flask tutorial foundation by **Albini (2021)**, with several extensions added to demonstrate a deeper understanding of how recommendation systems work.

**What this notebook covers:**
1. Loading the data and inspecting it
2. Building the SVD collaborative-filtering model
3. Generating personalized recommendations
4. Mood-based filtering
5. The content-based "similar movies" feature
6. The cold-start handler for new users
7. The hybrid slider (collaborative vs. content blend)
8. Submitting a new rating and seeing recommendations change

> **Note:** This notebook drives the recommendation *engine* (`myrecommender.py`) directly.
> The full graphical web app is launched separately from a terminal with `python app.py`.
> Both use the same underlying `MovieRecommender` class.

## 1. Setup

Make sure this notebook is running **inside the `cinematch` folder** so it can find `myrecommender.py`
and the `data/` directory. If the import below fails, check your working directory with `import os; os.getcwd()`.

In [20]:
# Confirming we are in the right directory
import os
print("Working directory:", os.getcwd())
print("Files here:", [f for f in os.listdir('.') if f.endswith('.py') or f == 'data'])

Working directory: C:\Users\durga\OneDrive\Desktop\MSAI Assignments\Summer 2026\AI for HCI\Recommendation Systems\cinematch
Files here: ['app.py', 'data', 'myrecommender.py']


In [21]:
# Import the recommendation engine
from myrecommender import MovieRecommender
import pandas as pd

print("Engine imported successfully.")

Engine imported successfully.


## 2. Load and Inspect the Data

The system uses two CSV files:
- **`movies.csv`** - 200 movies, each with a title and pipe-separated genre labels
- **`ratings.csv`** - ~20,000 ratings from 500 users on a 1-5 scale

Let's look at the raw data before the model touches it.

In [22]:
# Load the data into the recommender
r = MovieRecommender(n_factors=20)
r.load_data()

print(f"Total movies:  {len(r.movies_df)}")
print(f"Total ratings: {len(r.ratings_df)}")
print(f"Total users:   {r.ratings_df['user_id'].nunique()}")

Total movies:  214
Total ratings: 21136
Total users:   500


In [23]:
# Peek at the movies table
r.movies_df.head(10)

,movie_id,title,genres
0,1,The Shawshank Redemption,Drama
1,2,The Godfather,Crime|Drama
2,3,The Dark Knight,Action|Crime|Drama
3,4,Pulp Fiction,Crime|Drama
4,5,Schindlers List,Biography|Drama|History
5,6,The Lord of the Rings: Return of the King,Action|Adventure|Drama
6,7,12 Angry Men,Crime|Drama
7,8,Inception,Action|Adventure|Sci-Fi
8,9,Forrest Gump,Comedy|Drama|Romance
9,10,The Matrix,Action|Sci-Fi


In [24]:
# Peek at the ratings table
r.ratings_df.head(10)

,user_id,movie_id,rating
0,1,96,3.0
1,1,16,3.5
2,1,31,3.0
3,1,159,4.0
4,1,129,2.0
5,1,116,2.5
6,1,70,4.0
7,1,172,2.5
8,1,176,4.5
9,1,46,4.5


In [25]:
# How sparse is the rating matrix?
n_users = r.ratings_df['user_id'].nunique()
n_movies = r.movies_df['movie_id'].nunique()
n_ratings = len(r.ratings_df)
possible = n_users * n_movies
sparsity = 100 * (1 - n_ratings / possible)

print(f"Matrix is {n_users} users x {n_movies} movies = {possible:,} possible cells")
print(f"Only {n_ratings:,} are filled -> {sparsity:.1f}% sparse")
print("\nThis sparsity is the central challenge that SVD is designed to handle.")

Matrix is 500 users x 214 movies = 107,000 possible cells
Only 21,136 are filled -> 80.2% sparse

This sparsity is the central challenge that SVD is designed to handle.


## 3. Build the SVD Model

This is the core technique from the Albini tutorial. The `fit()` method:

1. Pivots the ratings into a **user x movie matrix**
2. **Mean-centers** each user's ratings (removes the bias of generous vs. harsh raters)
3. Runs **Singular Value Decomposition**: factorizing the matrix into `R = U · Σ · Vᵀ`
4. Keeps only the top **k = 20 latent factors** to capture the strongest taste patterns
5. Reconstructs the full matrix to predict ratings for movies each user hasn't seen

**My modification:** the user mean is computed only from *observed* ratings (not the zero-padded
matrix), and each user's predicted row is normalized back to a clean 1–5 scale. Without this fix,
predictions on a sparse matrix collapse to unrealistically low values.

In [26]:
# Fit the model (runs the SVD decomposition)
r.fit()

print("Model fitted.")
print(f"Predicted ratings matrix shape: {r.predicted_ratings.shape}")
print(f"Predicted rating range: {r.predicted_ratings.min():.2f} to {r.predicted_ratings.max():.2f}")

Model fitted.
Predicted ratings matrix shape: (500, 220)
Predicted rating range: 1.00 to 5.00


## 4. Generate Recommendations for a User

`recommend_for_user()` is where everything comes together. For a given user it:
- Pulls their SVD-predicted ratings for every unseen movie
- Computes a **content-based genre score** from their top-rated films
- Blends the two into a **hybrid score** (controlled by `alpha`)
- Applies a **confidence weight** so movies with very few ratings rank lower
- Returns the top-N with a plain-language **explanation** for each pick

In [27]:
# Recommendations for user 42
recs = r.recommend_for_user(user_id=42, n=8)

for i, rec in enumerate(recs, 1):
    print(f"{i}. [{rec['predicted_rating']}/5]  {rec['title']}  ({rec['genres']})")
    print(f"     -> {rec['explanation']}\n")

1. [3.92/5]  The Godfather Part II  (Crime|Drama)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

2. [3.86/5]  The Usual Suspects  (Crime|Drama|Mystery)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

3. [3.85/5]  Scarface  (Crime|Drama)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

4. [3.96/5]  Once Upon a Time in Hollywood  (Comedy|Drama)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

5. [3.76/5]  Hereditary  (Drama|Horror|Mystery)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

6. [3.73/5]  Chinatown  (Drama|Mystery|Thriller)
     -> Recommended based on your viewing pattern (similar to: Murder on the Orient Express, Lagaan)

7. [3.52/5]  American History X  (Crime|Drama)
     -> Recommended based on your viewing p

## 5. Mood-Based Filtering *(my extension)*

The base tutorial has no concept of mood. I added six mood categories, each mapped to a cluster
of genres. When a mood is selected, the system computes the same hybrid scores but then
**filters out** any movie that doesn't match the mood's genres before ranking.

Available moods:

In [28]:
print("Available moods:", r.get_available_moods())

Available moods: ['adventurous', 'thoughtful', 'lighthearted', 'tense', 'romantic', 'inspired']


In [29]:
# Same user, but now in a 'tense' mood -> thrillers, mystery, crime, horror
print("=== User 42, mood = 'tense' ===\n")
for rec in r.recommend_for_user(user_id=42, n=5, mood='tense'):
    print(f"[{rec['predicted_rating']}/5]  {rec['title']}  ({rec['genres']})")

=== User 42, mood = 'tense' ===

[3.92/5]  The Godfather Part II  (Crime|Drama)
[3.86/5]  The Usual Suspects  (Crime|Drama|Mystery)
[3.85/5]  Scarface  (Crime|Drama)
[3.76/5]  Hereditary  (Drama|Horror|Mystery)
[3.73/5]  Chinatown  (Drama|Mystery|Thriller)


In [30]:
# Same user, 'lighthearted' mood -> comedy, animation, family, romance
print("=== User 42, mood = 'lighthearted' ===\n")
for rec in r.recommend_for_user(user_id=42, n=5, mood='lighthearted'):
    print(f"[{rec['predicted_rating']}/5]  {rec['title']}  ({rec['genres']})")

=== User 42, mood = 'lighthearted' ===

[3.96/5]  Once Upon a Time in Hollywood  (Comedy|Drama)
[3.75/5]  Titanic  (Drama|Romance)
[4.3/5]  Coco  (Animation|Adventure|Family)
[3.55/5]  Before Sunrise  (Drama|Romance)
[3.42/5]  Gosford Park  (Comedy|Crime|Drama)


## 6. Content-Based: "Similar Movies" *(my extension)*

This feature ignores user history entirely. It represents each movie as a **binary genre vector**
and uses **cosine similarity** to find the films with the most overlapping genre profile. This is
the pure content-based half of the hybrid system.

In [31]:
# Find movies similar to The Dark Knight (movie_id = 3)
target = r.movies_df[r.movies_df['movie_id'] == 3].iloc[0]
print(f"Movies similar to: {target['title']} ({target['genres']})\n")

for s in r.similar_movies(movie_id=3, n=6):
    print(f"  {int(s['similarity_score']*100)}% match  ->  {s['title']}  ({s['genres']})")

Movies similar to: The Dark Knight (Action|Crime|Drama)

  100% match  ->  Heat  (Action|Crime|Drama)
  81% match  ->  The Godfather  (Crime|Drama)
  81% match  ->  Pulp Fiction  (Crime|Drama)
  81% match  ->  12 Angry Men  (Crime|Drama)
  81% match  ->  City of God  (Crime|Drama)
  81% match  ->  Top Gun: Maverick  (Action|Drama)


## 7. Cold-Start Handling *(my extension)*

What happens when a brand-new user with **no rating history** asks for recommendations?
SVD can't help — they aren't in the matrix. The system detects this and falls back to a
**popularity-weighted ranking** using `mean_rating × log(1 + rating_count)`, which balances
quality against how many people actually rated the film. Mood filtering still applies.

In [32]:
# User 9999 doesn't exist in the training data
print("=== New user (9999), mood = 'adventurous' ===\n")
for rec in r.recommend_for_user(user_id=9999, n=5, mood='adventurous'):
    print(f"[{rec['predicted_rating']}/5]  {rec['title']}")
    print(f"     -> {rec['explanation']}\n")

=== New user (9999), mood = 'adventurous' ===

[4.19/5]  KGF
     -> Highly rated by 66 viewers with an average of 4.2/5.0

[4.24/5]  Sholay
     -> Highly rated by 58 viewers with an average of 4.2/5.0

[4.16/5]  PK
     -> Highly rated by 59 viewers with an average of 4.2/5.0

[4.18/5]  Pushpa:The Rise
     -> Highly rated by 57 viewers with an average of 4.2/5.0

[3.81/5]  Arrival
     -> Highly rated by 85 viewers with an average of 3.8/5.0



## 8. The Hybrid Slider *(my extension)*

The `alpha` parameter controls the blend between the two recommendation philosophies:

| alpha | Behavior |
|-------|----------|
| `1.0` | Pure **SVD** collaborative filtering (the Albini baseline) |
| `0.0` | Pure **content-based** genre matching |
| `0.7` | Default 70/30 blend |

Watch how the recommendations shift for the same user as we slide alpha from collaborative to content.

In [33]:
user = 100

print(">>> alpha = 1.0  (pure collaborative filtering / SVD)")
for rec in r.recommend_for_user(user_id=user, n=5, alpha=1.0):
    print(f"    {rec['title']}  ({rec['genres']})")

print("\n>>> alpha = 0.0  (pure content-based / genre matching)")
for rec in r.recommend_for_user(user_id=user, n=5, alpha=0.0):
    print(f"    {rec['title']}  ({rec['genres']})")

>>> alpha = 1.0  (pure collaborative filtering / SVD)
    Ex Machina  (Drama|Sci-Fi|Thriller)
    Sherlock Holmes  (Action|Adventure|Crime)
    Oldboy  (Action|Drama|Mystery)
    Crazy Stupid Love  (Comedy|Drama|Romance)
    Midway  (Action|Drama|History)

>>> alpha = 0.0  (pure content-based / genre matching)
    Forrest Gump  (Comedy|Drama|Romance)
    Pans Labyrinth  (Drama|Fantasy|War)
    Love Actually  (Comedy|Drama|Romance)
    Crazy Stupid Love  (Comedy|Drama|Romance)
    500 Days of Summer  (Comedy|Drama|Romance)


## 9. Submitting a New Rating

The system accepts new ratings and marks the model for retraining. After adding ratings, the next
call to `recommend_for_user()` automatically refits the SVD model so the recommendations reflect
the new preference.

> Note: this writes to `data/ratings.csv`. If you want to keep the original data untouched while
> experimenting, make a backup copy of that file first.

In [34]:
# Check the rating count before
before = len(r.ratings_df)
print(f"Ratings before: {before}")

# User 100 rates a few sci-fi films highly
r.add_rating(user_id=100, movie_id=8,  rating=5.0)   # Inception
r.add_rating(user_id=100, movie_id=10, rating=5.0)   # The Matrix
r.add_rating(user_id=100, movie_id=12, rating=4.5)   # Interstellar

print(f"Ratings after:  {len(r.ratings_df)}  (+{len(r.ratings_df) - before})")

Ratings before: 21136
Ratings after:  21139  (+3)


In [35]:
# The model refits automatically on the next recommendation call
print("Updated recommendations for user 100:\n")
for rec in r.recommend_for_user(user_id=100, n=6):
    print(f"[{rec['predicted_rating']}/5]  {rec['title']}  ({rec['genres']})")

Updated recommendations for user 100:

[3.34/5]  Crazy Stupid Love  (Comedy|Drama|Romance)
[3.35/5]  Oldboy  (Action|Drama|Mystery)
[3.12/5]  Minari  (Drama)
[3.3/5]  Midway  (Action|Drama|History)
[3.52/5]  Ex Machina  (Drama|Sci-Fi|Thriller)
[2.96/5]  Roma  (Drama)


## 10. Inspecting Movie Statistics

A small utility for looking at the rating distribution of any single movie useful for
understanding why the confidence weighting treats some movies as more reliable than others.

In [36]:
# Stats for a few movies
for mid in [3, 8, 50]:
    title = r.movies_df[r.movies_df['movie_id'] == mid].iloc[0]['title']
    stats = r.get_movie_stats(mid)
    print(f"{title}")
    print(f"   ratings: {stats['count']}  |  mean: {stats['mean']}  |  std: {stats['std']}\n")

The Dark Knight
   ratings: 103  |  mean: 3.27  |  std: 1.13

Inception
   ratings: 93  |  mean: 3.63  |  std: 0.96

John Wick
   ratings: 80  |  mean: 3.58  |  std: 1.08



## Summary

This notebook demonstrated the full recommendation pipeline:

- **SVD collaborative filtering** (Albini foundation)- sections 3 & 4
- **Mood-based filtering** (extension)- section 5
- **Content-based genre similarity** (extension) - section 6
- **Cold-start handling** (extension)- section 7
- **Hybrid collaborative/content blending** (extension) - section 8
- **Live rating updates** (extension) - section 9

### To launch the full graphical web app
Open a terminal **in this folder** and run:

```bash
pip install -r requirements.txt
python app.py
```

Then visit **http://localhost:5000** in your browser to use the mood picker, hybrid slider,
movie cards, and rating modal through the GUI.

### Credit
Foundation: Albini, G. (2021). *Building a Movie Recommender Web App from Scratch with SVD and Flask.*
Extensions (hybrid scoring, mood filtering, confidence weighting, cold-start handling, explanations)
were added as part of this assignment.